> ⚠️ **Before you start:** This is a read-only course copy.
> Go to **File → Save a copy in Drive** right now, then continue working in *your* copy.
> Changes made here will not be saved.

In [ ]:
# Setup: install the course package (run this once per Colab session).
#
# Why the uninstall: the package version number never changes, so on a runtime
# that already has a copy, a plain install reports "already satisfied", skips the
# download, and leaves you running whatever code you installed last time.
#
# If you already ran an import cell before this one, use Runtime > Restart session
# afterwards. Python keeps the old module in memory even once the files are new.
!pip uninstall -q -y churn-pipeline
!pip install -q git+https://github.com/marceloacosta/churn-prediction-pipeline.git@main

# Chapter 7: LLM Integration

## Using Claude to Do the Boring Parts of ML Ops

Our pipeline has two places where a task is fundamentally about *understanding language*, something humans do effortlessly and rule-based code struggles with:

1. **Auto-Mapping (onboarding):** A new client uploads a CSV. Someone has to figure out what `MonthlyCharges`, `mrr`, or `amt_per_month` all mean. That's pattern recognition on natural language.

2. **Narrative Generation (output):** SHAP gives us `contract_type=month-to-month (+0.23)`. A business user needs: "This customer has no long-term commitment." That's translation from numbers to English.

Both tasks use Amazon Bedrock (Claude) via API, and neither one can crash the pipeline: if Bedrock fails, the call returns nothing and the caller carries on.

They are not equally harmless, though, and it is worth having the difference straight before you start. A missing narrative is cosmetic. The column reads `"N/A"` and the scores, risk tiers and SHAP reasons all ship as normal. A missing mapping stops that client dead: there is nothing to translate their columns with, and `load_mapping_config` raises on the file that is not there. Onboarding waits for a human.

So only one of these two is really optional. The **Failure Handling** section near the end works through both, including the failure that costs the most, which is not the LLM breaking but the LLM being confidently wrong.

### Cost

| Task | Typical Cost | When It Runs |
|------|-------------|-------------|
| Auto-Mapping | ~$0.01 per new client | Once, during onboarding |
| Narratives (50 customers) | ~$0.01-0.02 per batch | Each scoring run |

### What this chapter uses

Two modules do the work, one for each part of the chapter, and both have the same
three moving pieces: build a prompt, send it, parse what comes back. Everything else
imported here is the approval machinery from Chapter 1, which is what decides whether
the pipeline is allowed to act on any of it.

In [ ]:
# --- Part 1: auto-mapping. Build a prompt from a client's columns, send it, and
# turn the reply into a draft YAML somebody can review.
from churn_pipeline.llm.auto_mapping import (
    build_mapping_prompt,     # columns + sample rows + the standard schema -> one prompt
    call_bedrock_for_mapping, # sends it; returns mappings, or None if the call failed
    write_draft_yaml,         # writes the suggestions out as mapping.draft.yaml
    is_mapping_approved,      # cheap check: is there an approved mapping.yaml here?
    ColumnMapping,            # one suggestion: source column, target field, confidence, why
    _parse_mapping_response,  # Claude's JSON -> ColumnMapping objects. Underscore because
                              # it is internal; we call it directly to parse a saved reply.
)

# --- Part 2: narratives. Turn SHAP numbers for a batch of customers into English.
from churn_pipeline.llm.narrative_generator import (
    build_narrative_prompt,         # a batch of customers + their SHAP features -> one prompt
    call_bedrock_for_narratives,    # sends it; returns {customer_id: text}, or None
    parse_narrative_response,       # splits one reply back into per-customer narratives
    generate_narratives_for_batch,  # the whole loop: batching, and what to do on failure
    NarrativeRequest,               # one customer's prediction going in
    SYSTEM_PROMPT,                  # the house style every narrative has to follow
)

# --- The approval gate, from the config format built in chapter 1.
from churn_pipeline.mapping_config import (
    load_mapping_config,      # reads an approved mapping.yaml
    MappingNotApprovedError,  # ...and what it raises instead when handed a draft
)

# --- The standard field names every client's columns get translated into. This is
# the target vocabulary we hand the LLM.
from churn_pipeline.data_contract import STANDARD_SCHEMA

## First: are we calling Bedrock for real?

This chapter can run two ways.

**With credentials**, every prompt below goes to Claude on Amazon Bedrock and you
see what actually comes back, which is the point, because what comes back is not
identical every time.

**Without credentials**, the notebook falls back to saved responses and keeps going.
That fallback is a teaching device, and it is worth being precise about how far the
analogy goes. What the notebook shares with production is the detection: both LLM
functions return `None` when the call fails, and the caller branches on it. What differs
is the branch. Here it swaps in a saved answer so the chapter still has something to
explain. Production must not, because a canned narrative is indistinguishable from a
real one by the time it reaches a client. What production does instead is the last
section of this chapter.

If you want the live version, work through the **AWS credentials** setup page first:
one Bedrock API key in Colab Secrets, about five minutes. Otherwise just run on.

In [ ]:
import os


def load_aws_secrets():
    """Copy Colab Secrets into the environment, where boto3 looks for them.

    Returns the secrets it could not read, with the reason. Missing secrets are
    not an error here; the cells below fall back to saved responses.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return  # not in Colab: use whatever credentials this machine has

    missing = []
    for name in ("AWS_BEARER_TOKEN_BEDROCK", "AWS_DEFAULT_REGION"):
        try:
            os.environ[name] = userdata.get(name)
        except Exception as e:
            missing.append((name, type(e).__name__))
    return missing


missing = load_aws_secrets() or []

LIVE = bool(os.environ.get("AWS_BEARER_TOKEN_BEDROCK"))

if LIVE:
    print(f"Credentials found. Calling Bedrock for real in {os.environ.get('AWS_DEFAULT_REGION')}.")
elif any(reason == "NotebookAccessError" for _, reason in missing):
    # The secrets exist but this notebook is not allowed to read them. Every
    # notebook needs its own toggle, including copies you saved to Drive.
    print("Your secrets exist, but this notebook cannot read them.")
    print("Open the Secrets panel (key icon) and switch Notebook access on for:")
    for name, reason in missing:
        if reason == "NotebookAccessError":
            print(f"  - {name}")
    print("Then run this cell again. Falling back to saved responses for now.")
else:
    print("No credentials found. Running on saved responses.")
    print("(A stand-in so the chapter still runs. Production returns N/A instead;")
    print(" see Failure Handling near the end.)")

## Part 1: Auto-Mapping, or Teaching a Machine to Read Column Names

### The Problem

Every company calls their data something different:

| Company A | Company B | Company C | They All Mean... |
|-----------|-----------|-----------|------------------|
| MonthlyCharges | mrr | monthly_fee | monthly_charges |
| customerID | CustID | user_id | customer_id |
| Churn | left_service | is_churned | churn_label |

A rule-based approach would need an infinite dictionary of synonyms. An LLM handles this naturally because it *understands language*.

In [ ]:
# Simulate what a new client's CSV looks like
client_columns = ["CustID", "months_active", "MonthlyFee", "TotalSpend",
                   "left_service", "plan_type", "how_they_pay", "complaints"]

sample_rows = [
    {"CustID": "USR-7590", "months_active": 12, "MonthlyFee": 59.99,
     "TotalSpend": 719.88, "left_service": "no", "plan_type": "annual",
     "how_they_pay": "credit card", "complaints": 0},
    {"CustID": "USR-3344", "months_active": 2, "MonthlyFee": 99.99,
     "TotalSpend": 199.98, "left_service": "yes", "plan_type": "monthly",
     "how_they_pay": "bank transfer", "complaints": 4},
]

print("Client's raw columns:", client_columns)
print(f"\nSample row: {sample_rows[0]}")

In [ ]:
# Build the prompt we'd send to Claude
prompt = build_mapping_prompt(client_columns, sample_rows)

print("Prompt sent to Claude (first 800 chars):")
print("=" * 60)
print(prompt[:800])
print("...")
print(f"\n(Total prompt length: {len(prompt)} characters)")

In [ ]:
# The real call. `call_bedrock_for_mapping` sends the prompt, parses the JSON that
# comes back, and returns None if anything at all went wrong: no credentials, model
# not enabled, network, a reply that isn't valid JSON. It never raises.
SAVED_MAPPING_RESPONSE = """[
    {"source_column": "CustID", "target_field": "customer_id", "confidence": "high", "reasoning": "Contains 'ID' and values look like unique identifiers"},
    {"source_column": "months_active", "target_field": "tenure_months", "confidence": "high", "reasoning": "Directly describes duration in months"},
    {"source_column": "MonthlyFee", "target_field": "monthly_charges", "confidence": "high", "reasoning": "Monthly + Fee = monthly billing amount"},
    {"source_column": "TotalSpend", "target_field": "total_charges", "confidence": "high", "reasoning": "Cumulative spending"},
    {"source_column": "left_service", "target_field": "churn_label", "confidence": "medium", "reasoning": "Binary indicator of leaving, needs value mapping yes/no to 1/0"},
    {"source_column": "plan_type", "target_field": "contract_type", "confidence": "medium", "reasoning": "Describes contract duration category"},
    {"source_column": "how_they_pay", "target_field": "payment_method", "confidence": "high", "reasoning": "Payment method description"},
    {"source_column": "complaints", "target_field": "support_tickets", "confidence": "medium", "reasoning": "Complaint count likely correlates with support interactions"}
]"""

mappings = call_bedrock_for_mapping(prompt) if LIVE else None

if mappings is None:
    if LIVE:
        print("Bedrock did not answer. Falling back to the saved response.")
        print("The warning above says what Bedrock returned. To work out what to do")
        print("about it, run the verify cell on the AWS credentials page.\n")
    mappings = _parse_mapping_response(SAVED_MAPPING_RESPONSE)
else:
    print(f"Live from Bedrock: {len(mappings)} columns mapped.\n")

print("Claude's mapping suggestions:")
print("=" * 70)
print(f"{'Client Column':<18} {'\u2192 Standard Field':<20} {'Confidence':<12} Reasoning")
print("-" * 70)
for m in mappings:
    print(f"{m.source_column:<18} \u2192 {m.target_field:<18} {m.confidence:<12} {m.reasoning[:40]}")

### The Approval Workflow

The LLM's output is a **draft**. Three things have to happen before the pipeline will
touch it, and the next three cells do them one at a time.

**Why is a rename the approval?** Because the gate has to be something a person does on
purpose and anyone can check. No database, no approvals table, no state that can drift
out of sync with the file. You can answer "is this approved?" with `ls`.

The code enforces this. `load_mapping_config()` refuses a draft and raises
`MappingNotApprovedError`, and there is no flag to skip it. It checks two things,
because either one alone is easy to defeat by accident: the filename, and whether the
file still says `status: draft` inside.

#### Step 1: the LLM writes a draft

`write_draft_yaml` turns the suggestions into a file. Two things in it are the reason
this step exists at all:

- **`status: draft`** and the `.draft.yaml` filename. Both mark it unreviewed.
- **`confidence_scores`.** The model's own opinion of each guess, so a reviewer knows
  where to look first. `medium` is where the mistakes live.

Read the output. That file is what a human is being asked to check.

In [ ]:
import os

import yaml

# A real folder under the working directory. In Colab: the folder icon in the left
# sidebar, then client_configs/new_client.
config_dir = "client_configs/new_client"
os.makedirs(config_dir, exist_ok=True)
draft_path = os.path.join(config_dir, "mapping.draft.yaml")
approved_path = os.path.join(config_dir, "mapping.yaml")

write_draft_yaml(mappings, "new_client", draft_path)

print(f"The LLM wrote {draft_path}:")
print("-" * 64)
print(open(draft_path).read())

# And nothing downstream will touch it.
try:
    load_mapping_config(draft_path)
except MappingNotApprovedError as e:
    print(f"The pipeline refuses to load it:\n  {e}")

#### Step 2: a human reviews it

This is the step the whole chapter is built around, and in real life it is a person
opening the file in an editor.

Look at `value_mappings` in the output above: it is empty. Claude could see that
`left_service` holds `"yes"` and `"no"`, but nothing in the CSV tells it the pipeline
stores churn as `1` and `0`. The model had no way to avoid that gap, and it did not
flag it either, because the column mapping itself was right. Only somebody who knows
the contract can catch it.

That is the shape of this review in general: the model gets you most of the way and
cannot know what it cannot see. The cell below makes the edit in code so the chapter
can carry on. Nothing about it is automatic.

In [ ]:
config = yaml.safe_load(open(draft_path))

# The two edits a reviewer makes: fill in what the model could not know, then say out
# loud that a person has looked at it.
config["value_mappings"] = {"churn_label": {"yes": 1, "no": 0}}
config["status"] = "approved"

with open(draft_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("Two keys changed:")
print(f"  status:         'draft' -> {config['status']!r}")
print(f"  value_mappings: {{}} -> {config['value_mappings']}")
print("\nStill named .draft.yaml, so the pipeline still will not load it.")

#### Step 3: the rename is the approval

The file is correct now, and the pipeline still refuses it, because being correct and
being approved are two different claims. The rename is where a person makes the second
one.

Watch the three checks below. The draft does not count even after it was fixed;
`mapping.yaml` does not exist until the rename; and the moment it does, the gate opens.

In [ ]:
print("Before the rename:")
print(f"  mapping.draft.yaml approved? {is_mapping_approved(draft_path)}   (a draft never counts)")
print(f"  mapping.yaml approved?       {is_mapping_approved(approved_path)}   (does not exist yet)")

os.rename(draft_path, approved_path)

print("\nAfter the rename:")
print(f"  mapping.yaml approved?       {is_mapping_approved(approved_path)}")

loaded = load_mapping_config(approved_path)
print(f"\nload_mapping_config accepts it: client_id={loaded.client_id!r}, "
      f"{len(loaded.column_mappings)} columns mapped.")

# Renaming is not a way around reading. Someone who skipped step 2 would still have
# status: draft in the file. Shown on a throwaway copy so the real one stays approved.
lazy_path = os.path.join(config_dir, "renamed_but_unread.yaml")
config["status"] = "draft"
with open(lazy_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
try:
    load_mapping_config(lazy_path)
except MappingNotApprovedError as e:
    print(f"\nSkipping step 2 and renaming anyway does not work:\n  {e}")
os.remove(lazy_path)

print("\nOpen the folder icon in the Colab sidebar and you will find one file:")
print(f"  {approved_path}")
print("No mapping.draft.yaml beside it: step 3 renamed the draft rather than copying")
print("it, which is what makes 'is there a mapping.yaml here' a complete answer.")

### What Part 1 just did

| Step | What happened | What came out |
|------|---------------|---------------|
| Sample CSV | A new client's column names, plus two rows of data | `client_columns`, `sample_rows` |
| Build prompt | Wrapped those in instructions and the standard schema | a prompt of about 2,000 characters |
| Call Claude | Sent it to Bedrock, or used the saved reply | 8 `ColumnMapping` objects |
| Draft, review, rename | Wrote `mapping.draft.yaml`, filled in `value_mappings`, renamed it | `client_configs/new_client/mapping.yaml` |

**That last file is the actual output.** Everything the pipeline does for this client
from here on reads it to translate their column names into the standard schema. In
Chapter 1 you wrote a file like it by hand. What changed is who drafted it, and a human
still signs off before it counts.

## Part 2: Narrative Generation, or SHAP Numbers Turned Into English

### The Problem

After scoring, we have:
```
contract_type=month-to-month (+0.23); tenure_months=2 (+0.15); support_tickets=5 (+0.18)
```

A data scientist reads that and understands. A VP of Customer Success needs:

> "This customer has no long-term commitment and has contacted support 5 times in just 2 months. Month-to-month customers with high support volume are among the most likely to leave. Consider offering a discounted annual plan."

The LLM does this translation at scale.

### Where a batch comes from, and why it is a batch

**Who gets a narrative?** In Chapter 4 the pipeline scored every customer and gave each
one a probability and a tier: **high** at 0.70 and up, **medium** from 0.40, **low**
below that. That scored table is the input here.

Not everybody in it. The two customers below sit at 0.87 and 0.72, both high risk, and
those are the people somebody is going to phone this week. A paragraph about a customer
at 3% risk costs the same to generate and nobody will open it. On a 7,000-row file the
high tier is usually a few hundred rows, so filtering first is most of your cost saving
before you have tuned anything else.

**Why send them together?** The instructions, the feature definitions and the system
prompt are identical for every customer. Batch 50 into one call and you pay for that
preamble once. Make 50 separate calls and you pay for it 50 times, and wait through 50
round trips instead of one.

**Why 50 and not 500?** Three reasons, all pulling the same direction:

- A failed batch takes every customer in it down. At 50, an outage costs you 50
  narratives. At 500 it costs 500.
- Long prompts drift. Ask for 500 paragraphs in one go and the later ones get terse.
- The whole reply has to fit inside the output limit, and 50 paragraphs at 150 words
  each is already a large response.

`generate_narratives_for_batch` uses `batch_size=50` by default and slices the list for
you. Below we use two, so the prompt is short enough to read on screen.

In [ ]:
# Two customers, standing in for the high-risk slice of a scored run. Each carries what
# the model needs in order to write about it: how likely it is to leave, and which
# features pushed it there. `contribution` is the SHAP value from Chapter 4, meaning how
# much that feature moved this particular prediction, not how important it is overall.
batch = [
    NarrativeRequest(
        customer_id="CUST_001",
        churn_probability=0.87,
        risk_tier="high",
        top_shap_features=[
            {"feature": "contract_type", "contribution": 0.23},
            {"feature": "support_tickets", "contribution": 0.18},
            {"feature": "tenure_months", "contribution": 0.15},
        ],
    ),
    NarrativeRequest(
        customer_id="CUST_002",
        churn_probability=0.72,
        risk_tier="high",
        top_shap_features=[
            {"feature": "monthly_charges", "contribution": 0.19},
            {"feature": "contract_type", "contribution": 0.16},
            {"feature": "tenure_months", "contribution": 0.12},
        ],
    ),
]

# Our feature names are not English. Without these definitions the model has to guess
# what "tenure_months" means, and a guess in the prompt becomes a guess in the narrative.
feature_defs = {
    "contract_type": "Whether the customer is on month-to-month, annual, or two-year plan",
    "support_tickets": "Number of times the customer contacted support",
    "tenure_months": "How long they've been a customer",
    "monthly_charges": "What they pay each month",
}

prompt = build_narrative_prompt(batch, feature_definitions=feature_defs)

print(f"One prompt covering {len(batch)} customers, {len(prompt)} characters:")
print("=" * 60)
print(prompt[:1200])
print("...")

In [ ]:
# Same shape as the mapping call: real when we can, saved when we can't.
SAVED_NARRATIVE_RESPONSE = """CUSTOMER_ID: CUST_001
NARRATIVE: This customer is at very high risk of leaving. They have no long-term commitment (month-to-month plan), which means there's zero friction to cancel. They've also contacted support 5 times in just 2 months — a strong signal of frustration. With only 2 months of tenure, they haven't built any loyalty yet. Consider offering a discounted annual plan to lock them in, and escalate their open support issues immediately.

CUSTOMER_ID: CUST_002
NARRATIVE: This customer is paying significantly more than average ($110/month) on a month-to-month plan. High charges without a commitment create a "why am I paying this much?" moment. They're relatively new (4 months), so they're still in the window where switching costs are low. A loyalty discount or plan review could reduce their perceived cost and extend their stay."""

narratives = call_bedrock_for_narratives(prompt) if LIVE else None

if narratives is None:
    if LIVE:
        print("Bedrock did not answer. Falling back to the saved response.\n")
    narratives = parse_narrative_response(SAVED_NARRATIVE_RESPONSE, ["CUST_001", "CUST_002"])
else:
    print(f"Live from Bedrock: {len(narratives)} narratives.\n")

print("Generated narratives:")
print("=" * 60)
for cust_id, narrative in narratives.items():
    print(f"\n{cust_id}:")
    print(f"  {narrative[:200]}..." if len(narrative) > 200 else f"  {narrative}")

## What happens when it fails

This is the part the saved responses have been standing in for. On a real run that
loses Bedrock, nothing is substituted.

Both LLM functions return `None` instead of raising, so an outage does not take the
pipeline down. That is worth having. It is also the easy half of the problem, and this
chapter would be lying to you if it stopped there.

| Failure | What happens | What the client sees |
|---------|-------------|---------------|
| Auto-mapping fails | No draft. A human writes the YAML | Onboarding is blocked until they do. `load_mapping_config` raises on the missing file, so this client is not processed at all |
| Narrative generation fails | `narrative_explanation = "N/A"` | Scores, risk tiers and SHAP reasons ship as normal |

Only the second row is optional work. A client with no approved mapping goes nowhere,
and that is correct. The LLM saves you the typing; the requirement stands.

### Not crashing is the easy half

A step that swallows its failure and keeps going is good behaviour only if somebody
finds out. Right now the failure goes to `logger.warning`. In this notebook you see it
in red a few lines up, and it looks like the system working. In production it goes to a
log stream, and a warning nobody reads is the same as silence.

So the honest description of what we have built is: **it survives an LLM failure and
does not yet tell anyone.** Fine for the end of a chapter about calling a model. Not
fine to ship. The cell below runs a real outage, then does the thing the library leaves
to you.

In [ ]:
# Stand in for Bedrock during an outage. A client whose calls raise the way boto3 raises
# gets a realistic warning, rather than an error about the stub itself.
import botocore.exceptions


class UnreachableBedrock:
    def converse(self, **kwargs):
        raise botocore.exceptions.EndpointConnectionError(
            endpoint_url="https://bedrock-runtime.us-east-1.amazonaws.com"
        )


results = generate_narratives_for_batch(batch, boto3_client=UnreachableBedrock())

print("What came back:")
for cust_id, result in results.items():
    print(f"  {cust_id}: narrative={result.narrative!r}, success={result.success}")

# Every result carries a success flag. Nothing in the library adds them up, and a number
# nobody computes is a number nobody can act on.
failed = [r for r in results.values() if not r.success]
rate = len(failed) / len(results)
print(f"\n{len(failed)} of {len(results)} narratives failed ({rate:.0%}).")

# A run needs a rule. The number here is arbitrary on purpose: the point is that some
# number has to be written down, because carrying on regardless is also a choice.
FAILURE_BUDGET = 0.10

if rate > FAILURE_BUDGET:
    print(f"\nOver the {FAILURE_BUDGET:.0%} budget. This is where a real run stops and tells")
    print("somebody, rather than delivering a file full of N/A and letting the client be")
    print("the one who notices.")
else:
    print(f"\nUnder the {FAILURE_BUDGET:.0%} budget. Ship it, and record the rate anyway.")

### What is still missing

That was three lines, and the library does not do them for you. Nothing else in this
pipeline does either. A version you could put in front of a client also wants:

- **The failure rate in the run summary**, next to the AUC, so it is visible on a good
  day and not only when somebody goes looking.
- **An alert when the rate crosses the budget**, rather than a log line.
- **The reason, not only the count.** Throttling means retry. `AccessDeniedException`
  means an expired key, and no amount of retrying fixes that. Both land in the same
  `None` today.

Stage 10 is where this gets wired up. Until then, be precise about what you have:
narratives are best effort, and best effort without measurement is hope.

### The failure this chapter cannot catch

Everything above is the model failing to answer. The expensive failure is the model
answering, fluently, and being wrong.

A mapping that sends `complaints` to `support_tickets` when the client meant something
else does not raise. It renames the column, the row counts match, validation passes, and
the model trains on a feature that means something other than its name. You find out
when the predictions are quietly bad, months later, and the mapping is the last place
anyone looks.

No `try` block catches that. It is why Part 1 ends with a person renaming a file instead
of a confidence threshold. `confidence: high` is the model grading its own homework. The
rename is somebody signing it.

## The System Prompt: Controlling Claude's Output

The system prompt sets boundaries for what Claude writes. Four constraints:

- **Non-technical language.** No "SHAP values", no "feature importance".
- **Under 150 words.** Long enough to explain, short enough to read before a call.
- **Reference specific values.** "5 support tickets", with the number in it.
- **Plain English.** A VP should understand every word.

These are the difference between a paragraph somebody acts on and a paragraph somebody
skims past. Worth reading the actual text below and noticing how blunt it has to be.

In [ ]:
print("System prompt used for narrative generation:")
print("=" * 60)
print(SYSTEM_PROMPT)

## Key Takeaways

1. **Two LLM steps.** Auto-mapping at onboarding, narratives at output. Both are
   language problems, which is why rules were never going to cover them.
2. **The two are not equally optional.** A missing narrative is cosmetic. A missing
   mapping stops that client until a human writes one.
3. **Human-in-the-loop.** Auto-mapping produces a draft. `load_mapping_config` refuses
   to read it until somebody reviews it and renames it, and that gate is enforced in
   code rather than documented in a wiki.
4. **Surviving a failure is half the job.** Returning `None` keeps the pipeline up.
   Counting the failures and having a rule about the rate is what keeps you informed,
   and nothing does that for you yet.
5. **The dangerous failure is a confident wrong answer**, and no error handling catches
   it. That is the whole argument for the approval gate.
6. **Batching.** 50 customers per call, so the shared preamble is paid for once.
7. **Cost.** A few cents per run at these volumes.
8. **The system prompt does real work.** Under 150 words, no jargon, cite the numbers.

Next: Chapter 8 covers the AWS architecture, and how SageMaker Pipelines orchestrates
all of this into something that runs without a notebook.

---

*Source code: `src/churn_pipeline/llm/auto_mapping.py`, `src/churn_pipeline/llm/narrative_generator.py`*  
*Tests: `tests/unit/test_auto_mapping.py`, `tests/property/test_narrative.py`, `tests/unit/test_narrative.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*